# 94장 · 텍스트 정규화와 원문 해시

검색하기 좋게 정리한 문자열과 수집 당시의 원문은 용도가 다릅니다. 원문에 해시를 붙이면 정리된 문자열이 같아도 원본의 변경을 찾을 수 있습니다.

장비실 규정 사본을 정리하면서 원문 문자열을 UTF-8로 인코딩한 지문과 쪽·절 정보를 보존합니다. 줄바꿈을 변경했을 때 원본 지문과 처리된 문장이 각각 어떻게 달라지는지 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 조회할 원문의 판본과 텍스트 해시 남기기

93장의 조회 경로가 읽을 공개 원문을 가져옵니다. 줄바꿈을 정돈하기 전에 원문으로 해시를 만들면 같은 ID의 본문 교체를 확인할 수 있습니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter93.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter94-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter93.json"
sample_file = here / "chapter94-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['documents'])

교재 예시 chapter94-start.json
{'RULE-A': {'id': 'RULE-A', 'text': '점검 전 전원 표시등을 확인합니다.', 'source': '가상 장비실 규정 A', 'page': 1, 'section': '2.1', 'revised_on': '2026-11-01', 'version': 'rules-v1', 'equipment': '장비 A'}, 'RULE-B': {'id': 'RULE-B', 'text': '점검 전 고정 장치를 확인합니다.', 'source': '가상 장비실 규정 B', 'page': 2, 'section': '2.2', 'revised_on': '2026-11-01', 'version': 'rules-v1', 'equipment': '장비 B'}}


두 번째 줄은 읽은 파일의 `documents` 값입니다.

## 검색용 문자열과 원문 메타데이터

변환 예시　원문 RULE-A에는 “점검 전 전원 표시등을 확인합니다.”와 출처 1쪽 2.1절이 있습니다. 검색용 레코드는 공백을 정리한 본문, 원본에서 계산한 SHA-256, 원래의 식별자와 허용 그룹을 보관합니다. SHA-256은 내용이 그대로인지 대조할 문자열입니다. 출처가 믿을 만한지나 문장이 사실인지는 알려 주지 않습니다.


### 셀 01 · 원문 해시와 위치 메타데이터

원문에 손대지 않고 새 레코드를 만든 뒤 94장 제공 processed.json의 RULE-A와 같은 지문이 나오는지 읽습니다.

이 셀에서 새로 계산하는 지문은 raw의 UTF-8 본문에서 나옵니다. 제공된 processed.json의 지문은 비교 대상입니다. 파일을 직접 읽어 대조하지 않았다면 새로 계산한 값과 파일 내용을 확인한 일을 따로 적으세요.


In [1]:
from hashlib import sha256
raw = {'id': 'RULE-A',
 'equipment': '장비 A',
 'text': '점검 전 전원 표시등을 확인합니다.',
 'source': '가상 장비실 규정 A',
 'page': 1,
 'section': '2.1',
 'revised_on': '2026-11-01',
 'version': 'rules-v1',
 'groups': ['public', 'admin']}
prepared = dict(raw)
prepared["text"] = " ".join(raw["text"].split())
prepared["source_sha256"] = sha256(raw["text"].encode("utf-8")).hexdigest()
print(prepared["source_sha256"])
print(prepared["id"], prepared["section"], prepared["version"])

8336aeeaa41f3a89559a2364950ef88f7d6128639fb3dad46974a7be38ac2f42
RULE-A 2.1 rules-v1


지문은 8336aeea로 시작합니다. RULE-A, 2.1, rules-v1은 원본 그대로입니다. 본문을 먼저 정리한 뒤 지문을 계산하면 원본에 있던 추가 공백이나 줄바꿈을 놓칠 수 있습니다.

해시는 입력 바이트에서 계산합니다. 눈에 보이는 문장은 같아도 줄바꿈 표현이 다르면 원문 문자열을 UTF-8로 인코딩한 지문은 달라질 수 있습니다. 팀이 추적하려는 대상이 읽어 온 원문 문자열인지, 정규화한 본문인지에 따라 어느 단계의 지문을 저장할지 정합니다.


### SHA-256이 대조하는 입력의 범위

sha256에 전달되는 바이트는 raw의 text에서만 가져옵니다. 본문을 둔 채 section만 바꾸면 지문은 그대로입니다. 원문 위치까지 재현해야 할 때는 출력의 둘째 줄에 있는 식별자·절·판본도 따로 대조하세요.


## hashlib로 원문 변경 판별

재처리 메모 예시　교정 사본에는 점검 전 뒤에 줄바꿈이 들어갔습니다. 검색용 본문을 만들면 두 내용이 같지만 원문 지문은 달라집니다. 팀은 정리 전 원본의 차이를 보관하고 출처의 절 번호가 달라졌는지 확인한 뒤 새 판본을 발행합니다.

**검색용 문자열과 원문 해시**　빈칸 간격이 다른 두 원본의 지문이 다르게 표시됩니다. 정규화한 텍스트가 같더라도 원본을 그대로 비교한 결과는 다를 수 있습니다.

### 팀 참고 예시 · 줄바꿈과 원문 지문


In [2]:
edited = raw["text"].replace("전 ", "전\n")
print(sha256(edited.encode("utf-8")).hexdigest() == prepared["source_sha256"])
print(" ".join(edited.split()) == prepared["text"])

False
True


False 다음에 True가 나옵니다. 검색 결과가 같아 보여도 자료 담당자가 원본 변경을 확인할 수 있습니다.


#### 줄바꿈 하나가 바뀐 두 문서

<mark>해시는 입력 바이트를 비교하므로 검색용으로 정리한 본문이 같아도 원문 해시는 달라질 수 있습니다.</mark> 두 개의 불리언이 무엇을 비교했는지 먼저 읽으세요. 팀 자료에서는 원문 판본 확인용 지문과 검색용 문자열을 서로 다른 필드에 보관합니다.

선수 개념 · 16장 예외 조항과 청크 경계

## 재처리한 문서가 검색 테이블에 중복되는지 확인하기

RULE-A를 어제와 오늘 두 번 읽었습니다. 새 줄을 무조건 추가하면 같은 절이 검색 후보를 두 칸 차지할 수 있습니다. 여기서는 한 판본 안의 문서·절 조합을 저장 키로 정합니다.

### 셀 04 · 같은 문서·절·판본을 두 번 적재하기

앞 셀의 prepared는 공백을 정리한 본문과 원문 지문을 보관합니다. 임시 SQLite 테이블에 똑같은 레코드를 두 번 넣고 실제 남은 행을 확인합니다.

In [3]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as db:
    db.execute("""CREATE TABLE sections (
        doc TEXT, section TEXT, version TEXT, body TEXT, fingerprint TEXT,
        PRIMARY KEY(doc, section, version))""")
    fields = tuple(prepared[k] for k in ("id", "section", "version", "text", "source_sha256"))
    for _ in range(2):
        db.execute("INSERT OR IGNORE INTO sections VALUES (?, ?, ?, ?, ?)", fields)
    row = db.execute("SELECT doc, section, version, COUNT(*) FROM sections GROUP BY doc, section, version").fetchone()
    print(*row)

RULE-A 2.1 rules-v1 1


RULE-A 2.1 rules-v1이 한 행 남습니다. 이 삽입 방식은 같은 키에서 본문이 바뀌어도 무시합니다. 원문 지문이 달라지면 기존 값을 조용히 덮기 전에 새 판본인지 확인해야 합니다.

### 같은 ID 안에서 바뀐 내용 처리하기

자료 담당자는 개정 내용을 받았는데 판본 표기가 그대로인 경우를 등록 보류로 두고 두 지문을 기록합니다. 판본을 새로 부여했다면 두 행을 보관하고 현재 검색 대상 판본을 명시하세요. 검색 단계에서는 그룹 필터도 다시 적용합니다.

<mark>중복 적재를 막는 키가 원문 변경까지 알려 주지는 않습니다.</mark> 문서와 절이 같아도 판본이 다르면 과거 답변이 가리키는 원문을 찾기 위해 두 버전이 필요합니다.

95장 검색 결과는 레코드에 남긴 문서 ID·쪽·절·판본으로 원문을 다시 찾을 수 있어야 합니다.

## 팀 사례에 적용하기 · 조회할 원문의 판본과 텍스트 해시 남기기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
import hashlib
prepared_docs = []
for doc in incoming["documents"].values():
    stamp = hashlib.sha256(doc["text"].encode("utf-8")).hexdigest()
    prepared_docs.append({**doc, "search_text": " ".join(doc["text"].split()), "sha256": stamp})
new = {"quality_reference": incoming["quality_reference"], "prepared": prepared_docs, "paths": incoming["paths"]}
print([(doc["id"], doc["page"], doc["sha256"][:8]) for doc in prepared_docs])

[('RULE-A', 1, '8336aeea'), ('RULE-B', 2, 'ef43ae01')]


ID와 쪽수가 같아도 본문 한 글자를 고치면 해시가 달라집니다. 정규화한 텍스트와 원래 쪽 번호를 한 레코드에 두므로 다음 장에서 찾아온 문장이 어느 판본인지 확인할 수 있습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter94.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter94.json


95장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

95장에는 정리한 본문과 함께 ID·버전·출처 위치·허용 그룹을 넘기세요. 같은 ID와 판본의 내용을 다시 받을 때는 원본의 지문을 비교한 후 사용합니다.
